# Missing inputs, removed quantities and saved results

Missing factors do not become zero. Known quantities and contributions remain available. Temporary files below are automatically cleaned; existing files are refused by default.

In [ ]:
from dataclasses import replace

from teaching_case import (
    make_baseline,
    renovation_options,
    show_table,
    supplied_energy,
    teaching_settings,
)

from epsm_lca import assess_renovation

baseline = make_baseline()
windows, insulation = renovation_options(baseline)

step_1 = baseline.renovate(windows)
step_2 = step_1.renovate(insulation)
settings = teaching_settings(step_2)
energy = supplied_energy(baseline, step_1, step_2, annual_kwh=(10000, 9000, 8000))
factors = {
    identity: {m: f for m, f in values.items() if not m.startswith("C")}
    for identity, values in settings.material_factors.items()
}
partial = assess_renovation(
    step_2, replace(settings, material_factors=factors), energy=energy
)
assert not partial.complete
assert partial.stages[0].total_kgco2e is None
show_table(partial)
print(
    "Removed quantities still available:",
    partial.quantities(action="removed", by_material=True),
)

Restore the removal factors to complete the result. A saved report retains the actual choices, factors, their sources, building states, added and removed quantities, supplied energy and calculation version.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from epsm_lca import RenovationResult

complete = assess_renovation(step_2, settings, energy=energy)
assert complete.complete
with TemporaryDirectory() as directory:
    path = Path(directory) / "renovation.json"
    complete.save(path)
    restored = RenovationResult.load(path)
    assert restored == complete
    try:
        complete.save(path)
    except FileExistsError:
        print("Existing report correctly preserved")
    else:
        raise AssertionError("An existing report must not be overwritten silently")
    print(restored.summary())

Logging is optional and controlled by your notebook or application. The package reports events through Python logging; it never installs terminal or file output handlers. See the README for a setup example.